In [121]:
%pip install scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [122]:
import pandas as pd
import numpy as np


importing tools to do the train test split, make the pipeline, column transformer to easily convert the categorical and numerical features into onehot encoding and scaling repectively

In [123]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

lets load the csv file now

In [124]:
df=pd.read_csv("bank-additional-full.csv", sep=';')

In [125]:
df.shape

(41188, 21)

In [126]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

so there are no missing values at all

In [127]:
df.duplicated().sum()

np.int64(12)

there are 12 duplicated rows in here. lets see them

In [128]:
duplicates = df[df.duplicated]

In [129]:
duplicates

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
1266,39,blue-collar,married,basic.6y,no,no,no,telephone,may,thu,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
12261,36,retired,married,unknown,no,no,no,telephone,jul,thu,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.966,5228.1,no
14234,27,technician,single,professional.course,no,no,no,cellular,jul,mon,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no
16956,47,technician,divorced,high.school,no,yes,no,cellular,jul,thu,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no
18465,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.968,5228.1,no
20216,55,services,married,high.school,unknown,no,no,cellular,aug,mon,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.965,5228.1,no
20534,41,technician,married,professional.course,no,yes,no,cellular,aug,tue,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.966,5228.1,no
25217,39,admin.,married,university.degree,no,no,no,cellular,nov,tue,...,2,999,0,nonexistent,-0.1,93.200,-42.0,4.153,5195.8,no
28477,24,services,single,high.school,no,yes,no,cellular,apr,tue,...,1,999,0,nonexistent,-1.8,93.075,-47.1,1.423,5099.1,no
32516,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,...,4,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no


nothing wrong with the duplicated rows so ill drop them. 

In [130]:
df=df.drop_duplicates()

In [131]:
df.shape

(41176, 21)

now lets see if our data is imbalanced or not 

In [132]:
df["y"].value_counts()

y
no     36537
yes     4639
Name: count, dtype: int64

There are 36537 nos and only 4639 yes'. data is imbalanced

i will handle the imbalanced dataset after splitting; not now. 

In [133]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41176.00000,41176.000000,41176.000000,41176.000000,41176.000000,41176.000000,41176.000000,41176.000000,41176.000000,41176.000000
mean,40.02380,258.315815,2.567879,962.464810,0.173013,0.081922,93.575720,-40.502863,3.621293,5167.034870
std,10.42068,259.305321,2.770318,186.937102,0.494964,1.570883,0.578839,4.627860,1.734437,72.251364
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


now i will look at the features that we have : 
"age" "job" "marital" "education" "default" "housing" "loan" "contact" "month" "day_of_week" "duration" "campaign" "pdays" "previous" "poutcome" "emp.var.rate" "cons.price.idx" "cons.conf.idx" "euribor3m" "nr.employed" "y"

here duration represents the length of call that happened when the customer was called last time. here we are deciding  which customers to call before making the call. we cannot know the duration of the call before actually making it. 

also it is highly misleading, because the duration with 0 seconds willl always have answer 'no'

therefore we drop duration

In [134]:
df = df.drop(columns=['duration'])

In [135]:
for col in categorical_cols:
    print(df[col].value_counts())
    print('\n')

job
admin.           10419
blue-collar       9253
technician        6739
services          3967
management        2924
retired           1718
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64


marital
married     24921
single      11564
divorced     4611
unknown        80
Name: count, dtype: int64


education
university.degree      12164
high.school             9512
basic.9y                6045
professional.course     5240
basic.4y                4176
basic.6y                2291
unknown                 1730
illiterate                18
Name: count, dtype: int64


default
no         32577
unknown     8596
yes            3
Name: count, dtype: int64


housing
yes        21571
no         18615
unknown      990
Name: count, dtype: int64


loan
no         33938
yes         6248
unknown      990
Name: count, dtype: int64


contact
cellular     26135
telephone    15041
Name: coun

there are unknown values here. but i will consider them as a separate cztegory rather than removing them or imputing anything else. 

pdays:

In [136]:
print(df['pdays'].value_counts())

pdays
999    39661
3        439
6        412
4        118
9         64
2         61
7         60
12        58
10        52
5         46
13        36
11        28
1         26
15        24
14        20
8         18
0         15
16        11
17         8
18         7
22         3
19         3
21         2
25         1
26         1
27         1
20         1
Name: count, dtype: int64


999 pdays means customers were not contacted before. and there are ALOT of them

so what i will do is i will add onemore column named contactedaBefore which will be a binary column. it will have 1 if contacted before else 0 (pdays=999)

In [137]:
df['contacted_before'] = np.where(df['pdays'] == 999, 0, 1)


after doing this , we have separated the cusstomers into contactedbefore or not. but we need to chnage our og pdays column as well. we will change all the 999 values to 0. and keeping rest of it as it is. so the workflow will look something like: model will look if it was contacted before, from the new added column. if yes, thenn it will look howmany days ago from pdays column

In [138]:

df['pdays'] = np.where(df['pdays'] == 999, 0, df['pdays'])
df['y'] = df['y'].map({'yes': 1, 'no': 0})



In [139]:
df.shape


(41176, 21)

In [140]:
df.iloc[0]

age                          56
job                   housemaid
marital                 married
education              basic.4y
default                      no
housing                      no
loan                         no
contact               telephone
month                       may
day_of_week                 mon
campaign                      1
pdays                         0
previous                      0
poutcome            nonexistent
emp.var.rate                1.1
cons.price.idx           93.994
cons.conf.idx             -36.4
euribor3m                 4.857
nr.employed              5191.0
y                             0
contacted_before              0
Name: 0, dtype: object

In [141]:
df.iloc[-1]


age                                  74
job                             retired
marital                         married
education           professional.course
default                              no
housing                             yes
loan                                 no
contact                        cellular
month                               nov
day_of_week                         fri
campaign                              3
pdays                                 0
previous                              1
poutcome                        failure
emp.var.rate                       -1.1
cons.price.idx                   94.767
cons.conf.idx                     -50.8
euribor3m                         1.028
nr.employed                      4963.6
y                                     0
contacted_before                      0
Name: 41187, dtype: object

now the given dataset is order chronologically meaning the customers who were called first are literaly placed above. so we will not split the data in random order but instead split as first 80% and last 20%. basically we will train on more ""historical data"". 

also our y is in string format eith yes and no. so we convert it into 1 and 0

In [142]:

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]


X_train = train_df.drop(columns=['y'])
y_train = train_df['y']

X_test = test_df.drop(columns=['y'])
y_test = test_df['y']

#updating our numeric cols and categorical cols 
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")




Training Features Shape: (32940, 20)
Testing Features Shape: (8236, 20)


/var/folders/fy/zb78c__s7n317tdwn58j30kw0000gn/T/ipykernel_10629/2271391827.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()


using  columntransformer to do one hot encoding to our categorical features. and scale our numeric cols

In [143]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])



so now i will make our first pipeline. this will be our baseline. we still havent fixed our imbalanced dataset. so keeping it as our baseline will help us on next steps. 

we splitted dataset into 80-20 ratio. 

we added a contacted_before column to our dataframe. we converted string y into binary 1 and 0. 

we made a preprocessor to convert the categorical into one hot encode and numeric into proper scaling. 

In [144]:
from sklearn.linear_model import LogisticRegression


baseline_model = Pipeline([
    ('preprocessor', preprocessor),
  
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)) 
])


baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age','job','marital',...,'euribor3m','nr.employed','contacted_before']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``

In [145]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score


y_pred = baseline_model.predict(X_test)


y_prob = baseline_model.predict_proba(X_test)[:, 1] 


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report (Precision, Recall, F1):")
print(classification_report(y_test, y_pred))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC Score (Average Precision): {average_precision_score(y_test, y_prob):.4f}")

Confusion Matrix:
[[5126  571]
 [1710  829]]

Classification Report (Precision, Recall, F1):
              precision    recall  f1-score   support

           0       0.75      0.90      0.82      5697
           1       0.59      0.33      0.42      2539

    accuracy                           0.72      8236
   macro avg       0.67      0.61      0.62      8236
weighted avg       0.70      0.72      0.70      8236


ROC-AUC Score: 0.7479
PR-AUC Score (Average Precision): 0.5271


yes its terrible. since more nos are there in our dataset, it highly biased there. giving high recall and high f1 for class 0. 

now we will add class weight=balanced in our logistic regression model and see where that takes us. 

In [146]:

balanced_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)) 
])

balanced_model.fit(X_train, y_train)


y_pred_bal = balanced_model.predict(X_test)
y_prob_bal = balanced_model.predict_proba(X_test)[:, 1]



print("--- Balanced Model Evaluation ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_bal))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bal))
print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob_bal):.4f}")
print(f"PR-AUC Score (Average Precision): {average_precision_score(y_test, y_prob_bal):.4f}")

--- Balanced Model Evaluation ---
Confusion Matrix:
[[1862 3835]
 [ 387 2152]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.33      0.47      5697
           1       0.36      0.85      0.50      2539

    accuracy                           0.49      8236
   macro avg       0.59      0.59      0.49      8236
weighted avg       0.68      0.49      0.48      8236


ROC-AUC Score: 0.6981
PR-AUC Score (Average Precision): 0.5026


so it is catching the yes now because of our penalty but its givin too much yes making our precision down. 

lets see precision@500 and lift@500

In [149]:



base_rate = y_test.mean() 
expected_random_subscribers = base_rate * 500

print("1. Baseline (Blind Luck)")
print(f" this is base rate{base_rate:.4f}")
print(f"If a call center dialed 500 random people, they'd find ~{expected_random_subscribers:.1f} subscribers.\n")



ranking_df = pd.DataFrame({
    'Actual_Answer': y_test,
    'Probability_Score': y_prob_bal # Probabilities from the balanced model
})

ranking_df = ranking_df.sort_values(by='Probability_Score', ascending=False)


top_500_leads = ranking_df.head(500)


actual_subscribers_found = top_500_leads['Actual_Answer'].sum()
precision_at_500 = actual_subscribers_found / 500


lift_at_500 = precision_at_500 / base_rate

print(" 2. Our Balanced Model's Performance ")
print(f"Subscribers found in Top 500 calls: {actual_subscribers_found}")
print(f"Precision@500: {precision_at_500:.4f}")
print(f"Lift@500: {lift_at_500:.2f}x")
print(f"(This means our model is {lift_at_500:.2f} times more effective than calling at random!)")

1. Baseline (Blind Luck)
 this is base rate0.3083
If a call center dialed 500 random people, they'd find ~154.1 subscribers.

 2. Our Balanced Model's Performance 
Subscribers found in Top 500 calls: 301
Precision@500: 0.6020
Lift@500: 1.95x
(This means our model is 1.95 times more effective than calling at random!)


so we can conclude that our baseline model has got around 301 subscribers out of 500.